In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# ---------- SETTINGS ----------
initial_capital = 100000
lookback_period = "10y"   # keep smaller for speed

# PARAMETER GRID
stop_loss_list = [0.02, 0.04, 0.1]
drawdown_list = [0.01, 0.02, 0.1]
atr_mult_list = [1, 2, 10]

# ---------- LOAD NSE TICKERS ----------
def load_nse_tickers():
    df = pd.read_csv("data/EQUITY_L.csv")

    symbols = (
        df["SYMBOL"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    return [s + ".NS" for s in symbols if "&" not in s]

tickers = load_nse_tickers()[:2200]  # LIMIT for speed
print("Stocks used:", len(tickers))


# ---------- CLEAN DATA ----------
def clean_df(df):
    df = df.copy()

    # remove multi-index columns if any
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # keep only needed columns
    df = df[["Open", "High", "Low", "Close", "Volume"]]

    return df


# ---------- UT BOT ----------
def compute_utbot(df, atr_period=1, multiplier=1):

    df = clean_df(df)

    high = df["High"]
    low = df["Low"]
    close = df["Close"]

    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)

    atr = tr.rolling(atr_period).mean()

    df["atr"] = atr
    df["upper"] = close - multiplier * atr
    df["lower"] = close + multiplier * atr

    trend = [1]

    for i in range(1, len(df)):
        if close.iloc[i] > df["lower"].iloc[i - 1]:
            trend.append(1)
        elif close.iloc[i] < df["upper"].iloc[i - 1]:
            trend.append(-1)
        else:
            trend.append(trend[-1])

    df["trend"] = trend
    df["buy"] = (df["trend"] == 1) & (df["trend"].shift() == -1)
    df["sell"] = (df["trend"] == -1) & (df["trend"].shift() == 1)

    df["returns"] = close.pct_change()

    return df


# ---------- LOAD DATA ONCE ----------
all_data = {}

for ticker in tickers:
    try:
        df = yf.download(ticker, period=lookback_period, progress=False)

        if df.empty:
            continue

        df = compute_utbot(df)

        all_data[ticker] = df

    except:
        continue

print("Loaded:", len(all_data))


# ---------- BACKTEST FUNCTION ----------
def run_backtest(stop_loss_pct, drawdown_pct, atr_mult):

    cash = initial_capital
    open_positions = {}
    trade_log = []
    equity_curve = []

    all_dates = sorted(
        set(date for df in all_data.values() for date in df.index)
    )

    for current_date in all_dates:

        # ---------- SELL ----------
        for ticker in list(open_positions.keys()):

            df = all_data[ticker]

            if current_date not in df.index:
                continue

            row = df.loc[current_date]
            pos = open_positions[ticker]

            pos["highest"] = max(pos["highest"], row["Close"])

            stop_price = pos["entry"] * (1 - stop_loss_pct)
            atr_stop = pos["highest"] - (row["atr"] * atr_mult)

            drawdown = (pos["highest"] - row["Close"]) / pos["highest"]

            if row["Close"] <= stop_price:
                reason = "SL"
            elif row["Close"] <= atr_stop:
                reason = "ATR"
            elif drawdown >= drawdown_pct:
                reason = "DD"
            elif row["sell"]:
                reason = "Signal"
            else:
                continue

            exit_price = row["Close"]
            pnl = (exit_price - pos["entry"]) * pos["shares"]

            cash += pos["shares"] * exit_price

            trade_log.append(pnl)

            del open_positions[ticker]

        # ---------- BUY ----------
        for ticker, df in all_data.items():

            if ticker in open_positions:
                continue

            if current_date not in df.index:
                continue

            row = df.loc[current_date]

            if row["buy"]:

                allocation = cash * 0.05
                if allocation <= 0:
                    continue

                shares = allocation / row["Close"]

                cash -= allocation

                open_positions[ticker] = {
                    "entry": row["Close"],
                    "shares": shares,
                    "highest": row["Close"]
                }

        # ---------- EQUITY ----------
        total = cash
        for ticker, pos in open_positions.items():
            df = all_data[ticker]
            if current_date in df.index:
                total += pos["shares"] * df.loc[current_date]["Close"]

        equity_curve.append(total)

    # ---------- METRICS ----------
    if len(equity_curve) == 0:
        return None

    returns = pd.Series(equity_curve).pct_change().dropna()

    sharpe = 0
    if returns.std() != 0:
        sharpe = returns.mean() / returns.std() * np.sqrt(252)

    trades = np.array(trade_log)

    win_rate = 0
    avg_win = 0
    avg_loss = 0

    if len(trades) > 0:
        wins = trades[trades > 0]
        losses = trades[trades < 0]

        win_rate = len(wins) / len(trades)
        avg_win = wins.mean() if len(wins) > 0 else 0
        avg_loss = losses.mean() if len(losses) > 0 else 0

    return {
        "final_capital": equity_curve[-1],
        "return_%": (equity_curve[-1] / initial_capital - 1) * 100,
        "sharpe": sharpe,
        "win_rate": win_rate,
        "avg_win": avg_win,
        "avg_loss": avg_loss,
        "trades": len(trades)
    }


# ---------- OPTIMIZATION ----------
results = []

for sl in stop_loss_list:
    for dd in drawdown_list:
        for atr in atr_mult_list:

            print(f"Running SL={sl}, DD={dd}, ATR={atr}")

            res = run_backtest(sl, dd, atr)

            if res is None:
                continue

            res["stop_loss"] = sl
            res["drawdown"] = dd
            res["atr_mult"] = atr

            results.append(res)


# ---------- SAVE RESULTS ----------
df_results = pd.DataFrame(results)

df_results = df_results.sort_values("final_capital", ascending=False)

df_results.to_excel("optimization_results.xlsx", index=False)

print("\nSaved → optimization_results.xlsx")
print(df_results.head())

Stocks used: 2200


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ARISINFRA.NS"}}}
$ARISINFRA.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ARISINFRA.NS']: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['LTIM.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MEGASOFT.NS"}}}
$MEGASOFT.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['MEGASOFT.NS']: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['NGLFINE.NS']: TypeError("'NoneType' object is not subscriptable"

Loaded: 2194
Running SL=0.02, DD=0.01, ATR=1
Running SL=0.02, DD=0.01, ATR=2
Running SL=0.02, DD=0.01, ATR=10
Running SL=0.02, DD=0.02, ATR=1
Running SL=0.02, DD=0.02, ATR=2
Running SL=0.02, DD=0.02, ATR=10
Running SL=0.02, DD=0.1, ATR=1
Running SL=0.02, DD=0.1, ATR=2
Running SL=0.02, DD=0.1, ATR=10
Running SL=0.04, DD=0.01, ATR=1
Running SL=0.04, DD=0.01, ATR=2
Running SL=0.04, DD=0.01, ATR=10
Running SL=0.04, DD=0.02, ATR=1
Running SL=0.04, DD=0.02, ATR=2
Running SL=0.04, DD=0.02, ATR=10
Running SL=0.04, DD=0.1, ATR=1
Running SL=0.04, DD=0.1, ATR=2
Running SL=0.04, DD=0.1, ATR=10
Running SL=0.1, DD=0.01, ATR=1
Running SL=0.1, DD=0.01, ATR=2
Running SL=0.1, DD=0.01, ATR=10
Running SL=0.1, DD=0.02, ATR=1
Running SL=0.1, DD=0.02, ATR=2
Running SL=0.1, DD=0.02, ATR=10
Running SL=0.1, DD=0.1, ATR=1
Running SL=0.1, DD=0.1, ATR=2
Running SL=0.1, DD=0.1, ATR=10

Saved → optimization_results.xlsx
    final_capital      return_%    sharpe  win_rate        avg_win  \
0    1.390365e+09  1.390265

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from itertools import product

# ---------- SETTINGS ----------
initial_capital = 100000
max_positions = 10
lookback_period = "5y"

# ---------- LOAD NSE STOCKS ----------
def load_nse_tickers():
    df = pd.read_csv("data/EQUITY_L.csv")
    symbols = df["SYMBOL"].dropna().astype(str).str.strip().unique().tolist()

    tickers = []
    for s in symbols:
        if "&" not in s:
            tickers.append(s + ".NS")

    return tickers[:100]   # LIMIT for speed

tickers = load_nse_tickers()
print("Stocks Loaded:", len(tickers))


# ---------- UT BOT ----------
def compute_utbot(df, atr_period=1, multiplier=1):
    df = df.copy()

    df["tr"] = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    df["atr"] = df["tr"].rolling(atr_period).mean()

    # FIX: ensure Series not DataFrame
    df["upper"] = df["Close"].astype(float) - multiplier * df["atr"].astype(float)
    df["lower"] = df["Close"].astype(float) + multiplier * df["atr"].astype(float)

    trend = [1]

    for i in range(1, len(df)):
        if df["Close"].iloc[i] > df["lower"].iloc[i - 1]:
            trend.append(1)
        elif df["Close"].iloc[i] < df["upper"].iloc[i - 1]:
            trend.append(-1)
        else:
            trend.append(trend[-1])

    df["trend"] = trend
    df["buy"] = (df["trend"] == 1) & (df["trend"].shift() == -1)
    df["sell"] = (df["trend"] == -1) & (df["trend"].shift() == 1)

    df["volatility"] = df["Close"].pct_change().rolling(20).std()

    return df


# ---------- PRELOAD DATA (ONLY ONCE) ----------
all_data = {}

for ticker in tickers:
    try:
        df = yf.download(ticker, period=lookback_period, progress=False)

        if df.empty:
            continue

        df = compute_utbot(df)

        # Relaxed intrinsic logic (so trades happen)
        df["Intrinsic"] = df["Close"] * 1.2

        all_data[ticker] = df

    except:
        continue

print("Usable Stocks:", len(all_data))


# ---------- BACKTEST FUNCTION ----------
def run_backtest(stop_loss, drawdown, atr_mult):

    cash = initial_capital
    open_positions = {}
    trade_log = []
    equity_curve = []

    all_dates = sorted(set(d for df in all_data.values() for d in df.index))

    for current_date in all_dates:

        # ---------- SELL ----------
        for ticker in list(open_positions.keys()):
            df = all_data[ticker]

            if current_date not in df.index:
                continue

            row = df.loc[current_date]
            pos = open_positions[ticker]

            pos["highest"] = max(pos["highest"], row["Close"])

            stop_price = pos["entry"] * (1 - stop_loss)
            atr_stop = pos["highest"] - row["atr"] * atr_mult

            dd = (pos["highest"] - row["Close"]) / pos["highest"]

            if (
                row["Close"] <= stop_price
                or row["Close"] <= atr_stop
                or dd >= drawdown
                or row["sell"]
            ):
                exit_price = row["Close"]
                proceeds = pos["shares"] * exit_price
                profit = proceeds - pos["invested"]

                cash += proceeds

                trade_log.append(profit)

                del open_positions[ticker]

        # ---------- BUY ----------
        if len(open_positions) < max_positions:

            for ticker, df in all_data.items():

                if ticker in open_positions:
                    continue

                if current_date not in df.index:
                    continue

                row = df.loc[current_date]

                if pd.isna(row["volatility"]):
                    continue

                # RELAXED CONDITION (IMPORTANT)
                if not row["buy"]:
                    continue

                allocation = cash / (max_positions - len(open_positions))

                if allocation <= 0:
                    break

                shares = allocation / row["Close"]
                cash -= allocation

                open_positions[ticker] = {
                    "entry": row["Close"],
                    "highest": row["Close"],
                    "shares": shares,
                    "invested": allocation
                }

        # ---------- EQUITY ----------
        value = cash

        for ticker, pos in open_positions.items():
            df = all_data[ticker]

            if current_date in df.index:
                value += pos["shares"] * df.loc[current_date]["Close"]

        equity_curve.append(value)

    # ---------- METRICS ----------
    if len(equity_curve) < 2:
        return None

    final_capital = equity_curve[-1]

    returns = pd.Series(equity_curve).pct_change().dropna()

    sharpe = 0
    if returns.std() != 0:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)

    wins = [x for x in trade_log if x > 0]
    losses = [x for x in trade_log if x < 0]

    return {
        "stop_loss": stop_loss,
        "drawdown": drawdown,
        "atr_mult": atr_mult,
        "final_capital": final_capital,
        "return_%": (final_capital / initial_capital - 1) * 100,
        "sharpe": sharpe,
        "win_rate": len(wins) / len(trade_log) if trade_log else 0,
        "avg_win": np.mean(wins) if wins else 0,
        "avg_loss": np.mean(losses) if losses else 0,
        "trades": len(trade_log)
    }


# ---------- PARAMETER GRID (10+ VALUES EACH) ----------
stop_loss_vals = np.linspace(0.02, 0.10, 10)
drawdown_vals = np.linspace(0.01, 0.05, 10)
atr_vals = np.linspace(1, 5, 10)

param_grid = list(product(stop_loss_vals, drawdown_vals, atr_vals))

print("Total Trials:", len(param_grid))


# ---------- RUN OPTIMIZATION ----------
results = []

for i, (sl, dd, atr) in enumerate(param_grid):
    print(f"Running {i+1}/{len(param_grid)}")

    res = run_backtest(sl, dd, atr)

    if res is not None:
        results.append(res)
    else:
        # still record empty runs
        results.append({
            "stop_loss": sl,
            "drawdown": dd,
            "atr_mult": atr,
            "final_capital": initial_capital,
            "return_%": 0,
            "sharpe": 0,
            "win_rate": 0,
            "avg_win": 0,
            "avg_loss": 0,
            "trades": 0
        })


# ---------- SAVE RESULTS ----------
df_results = pd.DataFrame(results)

df_results = df_results.sort_values(by="final_capital", ascending=False)

df_results.to_excel("optimization_results.xlsx", index=False)

print("\n✅ Optimization Complete!")
print(df_results.head())

Stocks Loaded: 100
Usable Stocks: 0
Total Trials: 1000
Running 1/1000
Running 2/1000
Running 3/1000
Running 4/1000
Running 5/1000
Running 6/1000
Running 7/1000
Running 8/1000
Running 9/1000
Running 10/1000
Running 11/1000
Running 12/1000
Running 13/1000
Running 14/1000
Running 15/1000
Running 16/1000
Running 17/1000
Running 18/1000
Running 19/1000
Running 20/1000
Running 21/1000
Running 22/1000
Running 23/1000
Running 24/1000
Running 25/1000
Running 26/1000
Running 27/1000
Running 28/1000
Running 29/1000
Running 30/1000
Running 31/1000
Running 32/1000
Running 33/1000
Running 34/1000
Running 35/1000
Running 36/1000
Running 37/1000
Running 38/1000
Running 39/1000
Running 40/1000
Running 41/1000
Running 42/1000
Running 43/1000
Running 44/1000
Running 45/1000
Running 46/1000
Running 47/1000
Running 48/1000
Running 49/1000
Running 50/1000
Running 51/1000
Running 52/1000
Running 53/1000
Running 54/1000
Running 55/1000
Running 56/1000
Running 57/1000
Running 58/1000
Running 59/1000
Running 60

PermissionError: [Errno 13] Permission denied: 'optimization_results.xlsx'